# 🎵 Music Genre Recognition: Comprehensive Audio EDA & Visualization

This notebook provides an in-depth **Exploratory Data Analysis (EDA)** and **Audio Signal Visualization** for the 10-genre music dataset.

### 🎼 Covered Genres:
- `blues`, `classical`, `country`, `disco`, `hiphop`, `jazz`, `metal`, `pop`, `reggae`, `rock`

### 🔬 Audio Specifications:
- **Duration**: Standardized to 30.0 seconds
- **Sample Rate**: 16,000 Hz (Mono, 16-bit PCM WAV)
- **Provenance**: Open-Access Creative Commons & Public Domain historical recordings


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.titlesize'] = 14

GENRE_COLORS = {
    'blues': '#3498db', 'classical': '#9b59b6', 'country': '#f39c12',
    'disco': '#e91e63', 'hiphop': '#e74c3c', 'jazz': '#1abc9c',
    'metal': '#34495e', 'pop': '#2ecc71', 'reggae': '#16a085', 'rock': '#d35400'
}

DATASET_DIR = '.' if os.path.exists('metadata.csv') else 'kaggle_dataset'
meta_path = os.path.join(DATASET_DIR, 'metadata.csv')
df = pd.read_csv(meta_path)
print(f'Total tracks in dataset: {len(df)}')
df.head()

## 1. Class Distribution
Examine whether the classes are balanced and display overall dataset statistics.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
genre_counts = df['genre'].value_counts().sort_index()
colors = [GENRE_COLORS.get(g, '#666') for g in genre_counts.index]

bars = ax.bar(genre_counts.index, genre_counts.values, color=colors, edgecolor='black', alpha=0.85, width=0.6)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.1, f'{int(h)}', ha='center', va='bottom', fontweight='bold')

ax.set_title('Track Count per Music Genre', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Genre', fontweight='bold')
ax.set_ylabel('Number of Tracks', fontweight='bold')
ax.set_ylim(0, max(genre_counts.values) * 1.25)
plt.tight_layout()
plt.show()

## 2. Time-Domain Waveforms
Comparing the raw audio amplitudes $x(t)$ across genres illustrates dynamic range differences (e.g. classical has wide dynamics, while rock/metal has continuous high energy).

In [ ]:
genres = sorted(df['genre'].unique())
rows = (len(genres) + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(14, rows * 2.2), dpi=150, sharex=True)
axes = axes.flatten()

for idx, g in enumerate(genres):
    sample = df[df['genre'] == g].iloc[0]
    audio_path = os.path.join(DATASET_DIR, sample['file_path'])
    y, sr = librosa.load(audio_path, sr=16000, duration=30.0)
    t = np.linspace(0, len(y) / sr, len(y))
    
    ax = axes[idx]
    ax.plot(t, y, color=GENRE_COLORS.get(g, '#333'), alpha=0.8, linewidth=0.6)
    ax.set_title(f"{g.upper()}: {sample['title'][:30]}", fontsize=9.5, fontweight='bold', loc='left')
    ax.set_ylabel('Amp', fontsize=8)
    ax.set_ylim(-1.05, 1.05)

for i in range(len(genres), len(axes)):
    fig.delaxes(axes[i])

axes[len(genres)-1].set_xlabel('Time (s)', fontweight='bold')
if len(genres) > 1:
    axes[len(genres)-2].set_xlabel('Time (s)', fontweight='bold')
fig.suptitle('Time-Domain Audio Waveforms (30s at 16kHz)', fontsize=13, fontweight='bold', y=1.002)
plt.tight_layout()
plt.show()

## 3. Log-Mel Spectrograms
Converting audio to 64-Mel Spectrograms is the core representation used by modern Deep Learning models (CNNs, DINOv3 ViT, AST).

In [ ]:
fig, axes = plt.subplots(rows, 2, figsize=(14, rows * 2.5), dpi=150, sharex=True, sharey=True)
axes = axes.flatten()

for idx, g in enumerate(genres):
    sample = df[df['genre'] == g].iloc[0]
    audio_path = os.path.join(DATASET_DIR, sample['file_path'])
    y, sr = librosa.load(audio_path, sr=16000, duration=30.0)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=256, n_mels=64)
    S_dB = librosa.power_to_db(S, ref=np.max)
    
    ax = axes[idx]
    img = librosa.display.specshow(S_dB, sr=sr, hop_length=256, x_axis='time', y_axis='mel', ax=ax, cmap='magma')
    ax.set_title(f"{g.upper()}: {sample['title'][:30]}", fontsize=9.5, fontweight='bold', loc='left')
    ax.set_ylabel('Mel Hz', fontsize=8)

for i in range(len(genres), len(axes)):
    fig.delaxes(axes[i])

fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(img, cax=cbar_ax, format='%+2.0f dB')
fig.suptitle('Log-Mel Spectrograms Across Genres (64 Mels, n_fft=1024, hop=256)', fontsize=13, fontweight='bold', y=0.995)
plt.show()

## 4. Acoustic Features Extraction & Analysis
We extract standard audio descriptors:
- **Spectral Centroid**: Represents the 'brightness' or center of gravity of the spectrum.
- **Spectral Rolloff**: Frequency below which 85% of spectral energy lies.
- **Zero-Crossing Rate (ZCR)**: Indicator of noisiness / percussive content.
- **RMS Energy**: Root-mean-square loudness.

In [ ]:
features = []
for _, row in df.iterrows():
    p = os.path.join(DATASET_DIR, row['file_path'])
    if not os.path.exists(p):
        continue
    y, sr = librosa.load(p, sr=16000, duration=30.0)
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    rms = float(np.mean(librosa.feature.rms(y=y)))
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
    
    rec = {
        'genre': row['genre'],
        'spectral_centroid': centroid,
        'spectral_rolloff': rolloff,
        'zero_crossing_rate': zcr,
        'rms_energy': rms
    }
    for i, m in enumerate(mfccs):
        rec[f'mfcc_{i+1}'] = float(m)
    features.append(rec)

feat_df = pd.DataFrame(features)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=150)
sns.boxplot(data=feat_df, x='genre', y='spectral_centroid', palette=GENRE_COLORS, ax=axes[0, 0], hue='genre', legend=False)
axes[0, 0].set_title('Spectral Centroid (Frequency Brightness)', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=30)

sns.boxplot(data=feat_df, x='genre', y='spectral_rolloff', palette=GENRE_COLORS, ax=axes[0, 1], hue='genre', legend=False)
axes[0, 1].set_title('Spectral Rolloff (85% Energy)', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=30)

sns.boxplot(data=feat_df, x='genre', y='zero_crossing_rate', palette=GENRE_COLORS, ax=axes[1, 0], hue='genre', legend=False)
axes[1, 0].set_title('Zero-Crossing Rate (ZCR)', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=30)

sns.boxplot(data=feat_df, x='genre', y='rms_energy', palette=GENRE_COLORS, ax=axes[1, 1], hue='genre', legend=False)
axes[1, 1].set_title('RMS Loudness Energy', fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=30)

fig.suptitle('Acoustic Feature Comparison by Music Genre', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 2D Feature Space Visualization (PCA Projection)
Projecting MFCCs and acoustic features into a 2D space reveals natural musical clusters.

In [ ]:
feature_cols = [c for c in feat_df.columns if c != 'genre']
X = feat_df[feature_cols].values
y = feat_df['genre'].values

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
for g in sorted(np.unique(y)):
    idx = y == g
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1], label=g.upper(), color=GENRE_COLORS.get(g, '#666'), s=70, alpha=0.85, edgecolors='black')

ax.set_title(f'2D PCA Projection of Acoustic Features (Explained Var: {pca.explained_variance_ratio_.sum()*100:.1f}%)', fontweight='bold', fontsize=12)
ax.set_xlabel('Principal Component 1', fontweight='bold')
ax.set_ylabel('Principal Component 2', fontweight='bold')
ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()